# NLP Pipeline: Movie Review Analysis
**Name:** <your name here>  
**Roll No:** 57  
**Programme:** MCA | **Course:** Artificial Intelligence (MCS 3014) | **Session:** 2026-27 (Odd)  
**Group 5 - Assigned text type:** Movie Review

This notebook runs the preprocessing pipeline (Part I) and then text representation and sentiment analysis (Part II) on one movie review.

In [ ]:
# Step 0: install/import libraries and download the NLTK data we need
# Roll No: 57
!pip -q install nltk scikit-learn pandas

import re, string
import nltk
import pandas as pd

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
            "vader_lexicon"]:
    nltk.download(pkg, quiet=True)
print("Setup finished")

In [ ]:
# The text assigned to Group 5
review_text = (
    "The movie was entertaining from beginning to end and featured excellent "
    "performances by the lead actors. The visual effects and background music "
    "added to the overall experience, making several scenes memorable. Although "
    "the story slowed down slightly in the second half, the emotional ending "
    "made up for it. I would recommend this movie to anyone who enjoys action, "
    "drama, and inspiring stories."
)
print(review_text)

# PART I: Text Preprocessing

## Task 1: Identify and describe the text type
**What this step does:** Before cleaning anything we need to know what kind of text we are handling, because that decides how much noise to expect (slang, hashtags, emojis, etc.).

In [ ]:
n_sent = len(nltk.sent_tokenize(review_text))
n_words = len(review_text.split())
print("Text type      : Movie review (user-written, opinion based)")
print("Written style  : first person ('I would recommend'), subjective words")
print("Noise level    : low - no hashtags, emojis, URLs or slang")
print("Characters     :", len(review_text))
print("Rough words    :", n_words)
print("Sentences      :", n_sent)

**Description:** The text is a movie review. It gives the writer's personal opinion, mixes praise with a small complaint, and ends with a recommendation. It is clean, so it is well suited to sentiment analysis.

## Task 2: Sentence segmentation
**What this step does:** The paragraph is cut into separate sentences (mostly at full stops) so each can be analysed on its own.

In [ ]:
from nltk.tokenize import sent_tokenize
sentence_list = sent_tokenize(review_text)
for num, sent in enumerate(sentence_list, start=1):
    print(f"Sentence {num}: {sent}")
print("\nTotal sentences:", len(sentence_list))

## Task 3: Word tokenization
**What this step does:** Each sentence is split into tokens (words and punctuation marks). Tokens are the basic units that all later steps work on.

In [ ]:
from nltk.tokenize import word_tokenize
all_tokens = word_tokenize(review_text)
print(all_tokens)
print("\nTotal tokens:", len(all_tokens))

## Task 4: Lowercase, remove punctuation and numbers
**What this step does:** "Movie" and "movie" would otherwise count as two different words, so everything becomes lowercase. Punctuation and digits add no meaning here, so they are removed.

In [ ]:
text_lower = review_text.lower()
text_nopunct = text_lower.translate(str.maketrans("", "", string.punctuation))
text_clean = re.sub(r"[0-9]+", "", text_nopunct)
text_clean = re.sub(r"\s+", " ", text_clean).strip()

print("Lowercase     :", text_lower)
print()
print("Cleaned text  :", text_clean)
print()
clean_tokens = word_tokenize(text_clean)
print("Clean tokens  :", clean_tokens)
print("Digits found in original text:", re.findall(r"[0-9]+", review_text))

This review contains no numbers, so the number-removal step changed nothing, but it is still part of the pipeline.

## Task 5: Stop-word removal
**What this step does:** Very common words (the, was, to, and ...) are removed. They appear in almost every text and carry little meaning, so what remains are the content words.

In [ ]:
from nltk.corpus import stopwords
stop_set = set(stopwords.words("english"))

removed_words = [w for w in clean_tokens if w in stop_set]
content_tokens = [w for w in clean_tokens if w not in stop_set]

print("Stop words removed:", removed_words)
print()
print("Remaining tokens  :", content_tokens)
print()
print("Tokens before:", len(clean_tokens), "| after:", len(content_tokens))
print("Text after stop-word removal:", " ".join(content_tokens))

**Note:** The NLTK stop list also contains words like "although" and "not". Removing these can hide contrast or negation, which can matter for sentiment, so sentiment in Task 5 of Part II is run on the original sentences.

# PART II: Text Representation and Sentiment Analysis

## Task 1: Stemming
**What this step does:** Stemming chops word endings to get a crude root (for example *entertaining* becomes *entertain*). The stem may not be a real word.

In [ ]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()
stem_list = [stemmer.stem(w) for w in content_tokens]
stem_table = pd.DataFrame({"Original word": content_tokens, "Stemmed": stem_list})
display(stem_table)
print("Stemmed output:", " ".join(stem_list))

## Task 2: Lemmatization
**What this step does:** Lemmatization converts each word to its dictionary form using grammar knowledge (part of speech), so the result is always a real word. Slower than stemming, but more accurate.

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()

def to_wordnet_pos(treebank_tag):
    first = treebank_tag[0]
    if first == "J": return wordnet.ADJ
    if first == "V": return wordnet.VERB
    if first == "R": return wordnet.ADV
    return wordnet.NOUN

tagged = pos_tag(content_tokens)
lemma_list = [lemmatizer.lemmatize(w, to_wordnet_pos(t)) for w, t in tagged]

compare = pd.DataFrame({"Original": content_tokens,
                        "POS tag": [t for _, t in tagged],
                        "Stemmed": stem_list,
                        "Lemmatized": lemma_list})
display(compare)
print("Lemmatized output:", " ".join(lemma_list))

**Comparison:** Stemming is rough and can give non-words, while lemmatization keeps valid words. Compare the two columns in the table above and note the differences you see in your answer sheet.

## Task 3: Bag-of-Words (BoW)
**What this step does:** Text is turned into numbers by counting how often each word occurs, ignoring word order. Here each sentence is treated as one document, so we get a sentence-by-word count table.

In [ ]:
def prepare(sentence):
    s = re.sub(r"[^a-z\s]", "", sentence.lower())
    toks = [w for w in word_tokenize(s) if w not in stop_set]
    tg = pos_tag(toks)
    return " ".join(lemmatizer.lemmatize(w, to_wordnet_pos(t)) for w, t in tg)

docs = [prepare(s) for s in sentence_list]
for i, d in enumerate(docs, 1):
    print(f"Doc S{i}:", d)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
count_vec = CountVectorizer()
bow_matrix = count_vec.fit_transform(docs)

print("Vocabulary (word -> column index):")
print(count_vec.vocabulary_)
print()
bow_table = pd.DataFrame(bow_matrix.toarray(),
                         columns=count_vec.get_feature_names_out(),
                         index=[f"S{i}" for i in range(1, len(docs)+1)])
display(bow_table)
print("Total frequency of each word:")
print(bow_table.sum().sort_values(ascending=False))

## Task 4: TF-IDF
**What this step does:** TF-IDF gives a high weight to a word that is frequent in one sentence but rare across the other sentences, and a low weight to words spread everywhere. It is more informative than raw counts.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(docs)

tfidf_table = pd.DataFrame(tfidf_matrix.toarray().round(3),
                           columns=tfidf_vec.get_feature_names_out(),
                           index=[f"S{i}" for i in range(1, len(docs)+1)])
display(tfidf_table)

chosen_terms = ["movie", "excellent", "memorable", "recommend", "emotional", "inspiring"]
chosen_terms = [t for t in chosen_terms if t in tfidf_table.columns]
print("TF-IDF values for selected terms:")
display(tfidf_table[chosen_terms])

idf_table = pd.Series(tfidf_vec.idf_, index=tfidf_vec.get_feature_names_out()).round(3)
print("IDF of selected terms:")
print(idf_table[chosen_terms])

**Reading the result:** A word that appears in only one sentence has a higher IDF than a word that appears in several sentences. Use the table above to point out which selected terms score highest and explain why.

## Task 5: Sentiment classification
**What this step does:** We decide if the review is Positive, Negative or Neutral. First a tool (VADER) scores the text, then we justify the label using actual phrases from the review.

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

overall = analyzer.polarity_scores(review_text)
print("Overall scores:", overall)

for i, s in enumerate(sentence_list, 1):
    sc = analyzer.polarity_scores(s)
    print(f"S{i} compound = {sc['compound']:+.3f} | {s}")

c = overall["compound"]
if c >= 0.05:
    label = "Positive"
elif c <= -0.05:
    label = "Negative"
else:
    label = "Neutral"
print("\nFinal sentiment label:", label, f"(compound = {c:+.3f})")

### Justification (evidence from the text)
- **Positive phrases:** "entertaining from beginning to end", "excellent performances", "added to the overall experience", "memorable", "emotional ending made up for it", "I would recommend this movie", "inspiring stories".
- **Negative phrase:** only "the story slowed down slightly in the second half". The word *slightly* makes it mild, and the sentence itself says the ending made up for it.
- **Conclusion:** Praise clearly outweighs criticism and the review ends with a recommendation, so the overall sentiment is **Positive**. The VADER compound score above (threshold +0.05) supports this.

---
**End of notebook.** Roll No: 57 | Group 5 | Movie Review